# Part 8 integration candidate: non-publishing archive

Preserves the **exact** release tree of the audited integration dry run
`integration-20261108T052245Z-9bba8e` (run `correction-20261108T053328-949fcd76`; orchestration `a2c3171`;
pin `8aaa421`), or, if it is gone, reconstructs it and archives it **only if** a
strict scientific-equivalence gate passes against that audited session.

- **Preferred:** if the original runtime of that session is still connected, paste
  `colab/r1_part8_preserve_cell.py` into the ORIGINAL notebook and run it there instead.
- **This notebook** handles the case where the tree is gone. The reconstruction re-runs the
  unchanged, non-estimating dry run, so it requires `ALLOW_RECONSTRUCTION = True` (your decision).
- **Gate:** every exported machine report identical (CSVs byte for byte; JSON reports after
  removing only run_id/session/timestamps; gzip after decompression); acceptance A1–A15 passing,
  nothing unexplained, exit 4; the 465-file inventory identical except explicitly identified
  provenance carriers (figure sidecars, run records, regenerated tables' provenance columns);
  every figure and original file byte-identical. Anything else refuses the archive.
- **Writes only** `baseline/r1_part8_candidate_archive/<archive session>/` on Drive: `release.tar.gz`,
  `archive_inventory.csv`, `acceptance_report.json`, `archive_manifest.json`.
- No estimation, no publication, no push, no change to results, caches, certificates or releases.


In [ ]:
# @title Settings
REPOSITORY = "charlijomedia-dotcom/academia7"  # @param {type:"string"}
BRANCH = "part8-r1-rescore"  # @param {type:"string"}
ARCHIVE_TOOL_COMMIT = "d37f77f24a783ca13870fafc8c2c873eebf274be"  # @param {type:"string"}
ARCHIVE_TOOL_SHA256 = "969f3f427bc1100a83ae6161752f464f77e39c16a1068c16aea52e046024ba34"
ORCH_COMMIT = "a2c3171dbadc80f8821fdb91877d82622a1ff171"  # the audited orchestration; do not change
PIN_COMMIT = "8aaa4217536ab3cb942dd248dd19ece7838a8d60"  # the scientific pin; do not change
DRIVE_ROOT = "/content/drive/MyDrive/KillyPhD/Part8_K3_InSample_2026-10-29"  # @param {type:"string"}
EXPECTED_IDENTITY = "3413465b508bb844"
EXPECTED_ORIGINAL_PAYLOADS = 18964
ALLOW_RECONSTRUCTION = False  # @param {type:"boolean"}
WRITE_ARCHIVE = True  # @param {type:"boolean"}
USE_CONSTRAINTS = True
RUN_DRY_RUN = False           # set by the mode cell; True only for an authorised reconstruction
EXPORT_TO_DRIVE = False       # the dry run's report export is not used here
CONFIRM_PUBLISH = False       # there is no publication in this notebook; must stay False
WORK = "/content/r1_part8_integration"

## 1. Drive

In [ ]:
import os, re, sys, json, time, shutil, secrets, subprocess, importlib.util, hashlib
from pathlib import Path

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None
except ImportError:                       # outside Colab (used only to test this notebook)
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

assert CONFIRM_PUBLISH is False, "this notebook never publishes; CONFIRM_PUBLISH must be False"
DRIVE_ROOT = str(DRIVE_ROOT).strip().strip('"').strip("'").strip().rstrip("/")
assert DRIVE_ROOT.startswith("/content/drive/"), f"DRIVE_ROOT must be under /content/drive/: {DRIVE_ROOT!r}"
ORCH_COMMIT, PIN_COMMIT = str(ORCH_COMMIT).strip().lower(), str(PIN_COMMIT).strip().lower()
for _name, _sha in (("ORCH_COMMIT", ORCH_COMMIT), ("PIN_COMMIT", PIN_COMMIT)):
    assert re.fullmatch(r"[0-9a-f]{40}", _sha), f"{_name} must be a full 40-character SHA"
SESSION = f"integration-{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}-{secrets.token_hex(3)}"
WORK_DIR = Path(WORK)
WORK_DIR.mkdir(parents=True, exist_ok=True)
DRYRUN = WORK_DIR / "dryrun" / SESSION     # the driver's work folder (new for every session)
EXPORT = Path(DRIVE_ROOT) / "baseline/r1_part8_integration_dryrun" / SESSION   # the ONLY Drive write
os.environ.pop("CONFIRM_PUBLISH", None)
print("session:", SESSION, "| GitHub token:", "found" if GITHUB_TOKEN else "not set")

## 2. The archive tool and the audited export (both verified)

In [ ]:
# The archive tool, fetched read-only from ARCHIVE_TOOL_COMMIT and checked against its pinned sha256.
TOOL_DIR = Path(WORK) / "archive_tool" / ARCHIVE_TOOL_COMMIT[:12]
TOOL_DIR.mkdir(parents=True, exist_ok=True)
_tool_repo = Path(WORK) / "archive_tool" / "repo"
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if "GIT_ENV" in globals():
    _env = dict(GIT_ENV)
if not (_tool_repo / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", "--filter=blob:none",
                    f"https://github.com/{REPOSITORY}.git", str(_tool_repo)], check=True, env=_env)
subprocess.run(["git", "fetch", "origin", BRANCH], cwd=_tool_repo, check=True, env=_env)
_src = subprocess.run(["git", "show", f"{ARCHIVE_TOOL_COMMIT}:my-thesis/colab/r1_integration_archive.py"], cwd=_tool_repo,
                      check=True, capture_output=True, env=_env).stdout
assert hashlib.sha256(_src).hexdigest() == ARCHIVE_TOOL_SHA256, "archive tool does not match its pinned sha256"
(TOOL_DIR / "r1_integration_archive.py").write_bytes(_src)
sys.path.insert(0, str(TOOL_DIR))
import r1_integration_archive as arc
importlib.reload(arc)
AUDITED_EXPORT = Path(DRIVE_ROOT) / "baseline/r1_part8_integration_dryrun/integration-20261108T052245Z-9bba8e"
arc.verify_audited_export(AUDITED_EXPORT)
print("archive tool", ARCHIVE_TOOL_COMMIT[:12], "verified; audited export verified:", AUDITED_EXPORT)

## 3. Mode

In [ ]:
# Which mode: the audited tree on this runtime (preserve), or a reconstruction.
ORIGINAL_RELEASE = Path(WORK) / "dryrun" / arc.AUDITED_SESSION / "release"
if ORIGINAL_RELEASE.is_dir():
    MODE_RESOLVED = "preserve"
else:
    MODE_RESOLVED = "reconstruct"
    assert ALLOW_RECONSTRUCTION, ("the audited release tree is not on this runtime; a reconstruction "
                                  "re-runs the non-estimating dry run and needs ALLOW_RECONSTRUCTION = True")
    RUN_DRY_RUN = True
print("mode:", MODE_RESOLVED)

## 4a. Preserve (exact files)

In [ ]:
# PRESERVE: exact verification of the audited tree (465 files) against the audited inventory.
if MODE_RESOLVED == "preserve":
    GATE = arc.preserve_check(ORIGINAL_RELEASE, AUDITED_EXPORT)
    RELEASE_DIR, SOURCE_SESSION, SOURCE_RUN_ID = ORIGINAL_RELEASE, arc.AUDITED_SESSION, arc.AUDITED_RUN_ID
    SOURCE_ORCHESTRATION = arc.AUDITED_ORCHESTRATION
    print("preserve check passed:", GATE["files"], "files, all identical to the audited inventory")

## 4b. Reconstruct: checkouts at the audited commits

In [ ]:
if MODE_RESOLVED == "reconstruct":
    exec(compile('SECRETS = WORK_DIR / "secrets"\nSECRETS.mkdir(exist_ok=True)\nGIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0")\nif GITHUB_TOKEN:                          # read access only; nothing here pushes\n    askpass = SECRETS / "git_askpass.sh"\n    askpass.write_text(\'#!/bin/sh\\ncase "$1" in\\n  Username*) echo "x-access-token" ;;\\n\'\n                       \'  *) echo "$THESIS_GIT_TOKEN" ;;\\nesac\\n\')\n    askpass.chmod(0o700)\n    GIT_ENV.update(GIT_ASKPASS=str(askpass), THESIS_GIT_TOKEN=GITHUB_TOKEN)\nREPO = WORK_DIR / "repo"                  # orchestration checkout\nPINNED = WORK_DIR / "pinned"              # scientific-code checkout (worktree)\nurl = f"https://github.com/{REPOSITORY}.git"\nif not (REPO / ".git").exists():\n    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO)], check=True, env=GIT_ENV)\nsubprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True, env=GIT_ENV)\nsubprocess.run(["git", "checkout", "-q", "--detach", ORCH_COMMIT], cwd=REPO, check=True)\nif not PINNED.exists():\n    subprocess.run(["git", "worktree", "add", "-q", "--detach", str(PINNED), PIN_COMMIT],\n                   cwd=REPO, check=True)\ndef _head(path):\n    return subprocess.run(["git", "rev-parse", "HEAD"], cwd=path, capture_output=True,\n                          text=True, check=True).stdout.strip()\ndef _clean(path):\n    return not subprocess.run(["git", "status", "--porcelain"], cwd=path, capture_output=True,\n                              text=True, check=True).stdout.strip()\nassert _head(REPO) == ORCH_COMMIT and _clean(REPO), "orchestration checkout not at ORCH_COMMIT or dirty"\nassert _head(PINNED) == PIN_COMMIT and _clean(PINNED), "scientific checkout not at PIN_COMMIT or dirty"\nsubprocess.run(["git", "merge-base", "--is-ancestor", PIN_COMMIT, ORCH_COMMIT], cwd=REPO, check=True)\nCODE = PINNED / "my-thesis" / "code"      # every estimation and table comes from here\nORIGINAL = PINNED / "my-thesis" / "output"\nORCH = REPO / "my-thesis" / "colab"       # orchestration helpers only\nassert (ORCH / "thesis_colab.py").read_bytes() == (PINNED / "my-thesis" / "colab" / "thesis_colab.py").read_bytes()\nsys.path.insert(0, str(ORCH))\nimport r1_tierb as tb\nspec = importlib.util.spec_from_file_location("thesis_colab", ORCH / "thesis_colab.py")\ntc = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(tc)\nprint("orchestration", ORCH_COMMIT[:12], "| scientific code", PIN_COMMIT[:12])\n# Reference releases (read only, as git objects): Tier A (292227e) and the R1 release the claim\n# matrix was written against (0573a8c).\nfor _branch in (\'part8-corrected-results/r1-tier-a-20261103T063749Z\', \'part8-corrected-results/r1-20261101\'):\n    subprocess.run(["git", "fetch", "origin", _branch], cwd=REPO, check=True, env=GIT_ENV)\nfor _commit in ("292227e0904bb9a857c76d633031c39fbd463eae", "0573a8c"):\n    subprocess.run(["git", "cat-file", "-e", _commit + "^{commit}"], cwd=REPO, check=True)\nassert _clean(REPO) and _clean(PINNED), "a checkout became dirty"\nprint("reference releases available")', "<reconstruction>", "exec"), globals())
else:
    print("preserve mode: not needed")

## 4c. Reconstruct: approved environment, frozen data, cache identity

In [ ]:
if MODE_RESOLVED == "reconstruct":
    exec(compile('VENV = WORK_DIR / "venv311"\nLOCK = CODE / "requirements.lock.txt"\nCONSTRAINTS = CODE / "requirements.constraints.txt" if USE_CONSTRAINTS else None\nPY = VENV / "bin" / "python"\ntry:\n    tc.verify_environment(PY, LOCK, constraints=CONSTRAINTS)\nexcept Exception:\n    PY = tc.create_environment(VENV, LOCK, constraints=CONSTRAINTS)\nFOUND = tc.verify_environment(PY, LOCK, constraints=CONSTRAINTS)\nRUN_ENV = dict(os.environ, PYTHONUNBUFFERED="1", MPLBACKEND="Agg")\nRUN_ENV.pop("THESIS_GIT_TOKEN", None)\ncheck = subprocess.run([str(PY), "-u", "code/run_all.py", "--data-only"], cwd=PINNED / "my-thesis",\n                       env=RUN_ENV, capture_output=True, text=True)\nprint(check.stdout[-1200:], check.stderr[-1200:])\nassert check.returncode == 0, "frozen-data verification failed; nothing was started"\nFROZEN_SHA = tc.sha256(PINNED / "my-thesis" / "data" / "frozen" / "thesis_input_2026-10-05.csv")\nIDENTITY = tc.cache_identity(PY, CODE, "baseline")\nassert IDENTITY == EXPECTED_IDENTITY, f"cache identity {IDENTITY} != {EXPECTED_IDENTITY}"\nprint(f"Python {FOUND[\'python\']}; {len(FOUND[\'packages\'])} packages verified; identity {IDENTITY}")', "<reconstruction>", "exec"), globals())
else:
    print("preserve mode: not needed")

## 4d. Reconstruct: the original fit cache (read only)

In [ ]:
if MODE_RESOLVED == "reconstruct":
    exec(compile('# The complete ORIGINAL fit cache, read only, onto local disk (nothing is written on Drive).\nDRIVE = Path(DRIVE_ROOT) / "baseline"\nORIGINAL_CACHE = WORK_DIR / "original_cache"\nFOLDERS = list(tc.archive_folders(DRIVE, IDENTITY))\nif not (ORIGINAL_CACHE / "fits").exists():\n    for folder in FOLDERS:\n        tc.IncrementalArchive(ORIGINAL_CACHE / "fits", folder).restore(folder=folder)\nPAYLOADS = sum(1 for _ in (ORIGINAL_CACHE / "fits").glob("*/*.json"))\nassert PAYLOADS == EXPECTED_ORIGINAL_PAYLOADS, f"{PAYLOADS} original payloads, expected {EXPECTED_ORIGINAL_PAYLOADS}"\nTIER_A_ARCHIVE = Path(DRIVE_ROOT) / tb.TIER_A_ARCHIVE_RELATIVE\nassert TIER_A_ARCHIVE.is_file(), f"missing {TIER_A_ARCHIVE}"\nprint(f"{PAYLOADS} original payloads restored from {len(FOLDERS)} archive folder(s); Tier A archive found")', "<reconstruction>", "exec"), globals())
else:
    print("preserve mode: not needed")

## 4e. Reconstruct: the unchanged dry run (no estimation, no publication)

In [ ]:
DRY_EXIT = None
if MODE_RESOLVED == "reconstruct":
    exec(compile('# Non-estimating and non-publishing. Exit 0 = accepted; 4 = accepted, REVIEW REQUIRED; 1 = REFUSED.\nif not RUN_DRY_RUN:\n    print("RUN_DRY_RUN is False")\nelse:\n    assert not DRYRUN.exists(), f"{DRYRUN} exists"\n    command = [str(PY), "-u", str(ORCH / "r1_part8_integration.py"), "--repo", str(REPO),\n               "--code-dir", str(CODE), "--drive-root", DRIVE_ROOT,\n               "--cache-root", str(ORIGINAL_CACHE), "--tier-a-archive", str(TIER_A_ARCHIVE),\n               "--orch-commit", ORCH_COMMIT, "--work", str(DRYRUN)]\n    for _i, _folder in enumerate(FOLDERS):     # the original-cache archives on Drive stay unchanged\n        command += ["--protect", f"original_cache_archive_{_i}={_folder}"]\n    RUN_ENV_DRY = dict(RUN_ENV)\n    RUN_ENV_DRY.pop("CONFIRM_PUBLISH", None)\n    started = time.time()\n    dry = subprocess.run(command, cwd=str(CODE), env=RUN_ENV_DRY, capture_output=True, text=True)\n    (DRYRUN / "dryrun_report").mkdir(parents=True, exist_ok=True)\n    (DRYRUN / "dryrun_report" / "console.log").write_text(dry.stdout + "\\n--- stderr ---\\n" + dry.stderr)\n    print(dry.stdout[-8000:])\n    print(dry.stderr[-3000:])\n    DRY_EXIT = dry.returncode\n    print(f"dry run exit {DRY_EXIT} after {(time.time() - started) / 60:.1f} min")\n    status = [l for l in dry.stdout.splitlines() if l.startswith("DRY_RUN_STATUS: ")]\n    print(status[-1] if status else "DRY_RUN_STATUS: unknown (see console.log)")\n    print("A completed or accepted dry run is NOT a scientifically approved result; nothing was published.")', "<reconstruction>", "exec"), globals())
else:
    print("preserve mode: not needed")

## 4f. Reconstruct: the equivalence gate

In [ ]:
# RECONSTRUCT: the strict equivalence gate against the audited session (refuses on any other difference).
if MODE_RESOLVED == "reconstruct":
    assert DRY_EXIT is not None, "the reconstruction did not run"
    _acc = json.loads((DRYRUN / "dryrun_report" / "acceptance.json").read_text())
    GATE = arc.equivalence_gate(DRYRUN, AUDITED_EXPORT, exit_code=DRY_EXIT, new_session=SESSION,
                                new_run_id=_acc["run_id"], published_dir=ORIGINAL)
    RELEASE_DIR, SOURCE_SESSION, SOURCE_RUN_ID = DRYRUN / "release", SESSION, _acc["run_id"]
    SOURCE_ORCHESTRATION = ORCH_COMMIT
    print(json.dumps(GATE["checks"]), "| provenance-only files:", len(GATE["inventory"]["provenance_only"]))

## 5. Archive to a new Drive folder

In [ ]:
# Write the archive to a NEW Drive folder (refuses if it exists), then re-read and verify it.
ARCHIVE_SESSION = f"archive-{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}-{secrets.token_hex(3)}"
IDENTITY = arc.run_identity(mode=GATE["mode"], archive_session=ARCHIVE_SESSION,
                            archive_tool_commit=ARCHIVE_TOOL_COMMIT,
                            orchestration_commit=SOURCE_ORCHESTRATION, scientific_commit=arc.SCIENTIFIC_PIN,
                            run_id=SOURCE_RUN_ID, source_session=SOURCE_SESSION)
if not WRITE_ARCHIVE:
    print("WRITE_ARCHIVE is False; gate report:", json.dumps(GATE)[:2000])
else:
    MANIFEST = arc.write_archive(RELEASE_DIR, DRIVE_ROOT, ARCHIVE_SESSION, IDENTITY, GATE)
    DEST = Path(DRIVE_ROOT) / arc.ARCHIVE_PARENT / ARCHIVE_SESSION
    print(json.dumps(arc.verify_archive(DEST)))
    print(f"archived {MANIFEST['files']} files to {DEST}; release.tar.gz sha256 {MANIFEST['release_tar_sha256']}")
    print("Nothing was published. Please send this folder name to Claude:", ARCHIVE_SESSION)